# Koddan Kasa · Levha 02 · Dünya her 2 ms'de bir hesaplanır

Videodaki deney: 5 cm'lik küp 1 m yükseklikten 1 cm'lik bir tablaya bırakılıyor. Tek değişen sayı MuJoCo'nun **zaman adımı** (`timestep`).
Bizim sonuçlar (MuJoCo 3.17, 1,5 sn simülasyon): 2–8 ms kaldı · 10 ms geçti · 12 kaldı · 15 geçti · 20 kaldı · 25, 30, 40, 50 ms geçti.

Neden? Simülasyon zamanı adım adım atlar ve çarpışmaya yalnız bu anlarda bakar. Küp tablayla ilk örtüştüğü adımda merkezi tablanın ortasının **altındaysa** fizik onu aşağı iter. 25 ms ve üstünde temas hiç görülmeden geçer.

## 1 · Kurulum

In [ ]:
!pip -q install mujoco==3.17.0

## 2 · Sahne: zemin, 1 cm tabla, 5 cm küp

In [ ]:
import mujoco, numpy as np
XML = """
<mujoco>
  <worldbody>
    <geom name="floor" type="plane" size="2 2 0.1"/>
    <geom name="table" type="box" size="0.12 0.12 0.005" pos="0 0 0.295"/>
    <body name="cube" pos="0 0 1.30">
      <freejoint/>
      <geom type="box" size="0.025 0.025 0.025" mass="0.1"/>
    </body>
  </worldbody>
</mujoco>
"""

## 3 · Tek satır: timestep

In [ ]:
def drop(dt_ms, sim_sec=1.5):
    m = mujoco.MjModel.from_xml_string(XML)
    m.opt.timestep = dt_ms / 1000          # ← değiştirdiğimiz tek sayı
    d = mujoco.MjData(m)
    z = []
    while d.time < sim_sec:
        mujoco.mj_step(m, d)
        z.append(d.body("cube").xpos[2])
    return np.array(z)

for dt in [2, 4, 6, 8, 10, 12, 15, 20, 25, 30, 40, 50]:
    z = drop(dt)
    print(f"dt {dt:2d} ms · {1000//dt:3d} adım/sn · son yükseklik {z[-1]*100:5.1f} cm ·", "GEÇTİ" if z[-1] < 0.25 else "kaldı")

> Not: Bu sahne sadeleştirilmiş (robot kol yok). Yerelde (MuJoCo 3.17) videodaki 12 sonucun hepsini aynen verdi: 2–8 kaldı · 10 geçti · 12 kaldı · 15 geçti · 20 kaldı · 25–50 geçti.


## 4 · Adım adım: 10 ms'de küp nerede?

In [ ]:
z = drop(10)
for k in range(40, 50):
    print(f"t = {(k+1)*10/1000:.2f} sn · küp merkezi {z[k]*100:6.2f} cm   (tabla ortası 29,5 cm)")

## Dene
- Tablayı kalınlaştır (`size="0.12 0.12 0.02"`): hangi dt'lerde geçiyor?
- Küpü daha yükseğe koy (hız artar): eşik değişiyor mu?
- Sence robot eğitim simülasyonlarında adım kaç ms? Yorumlara yaz.